# 03 — Baseline 2: BiLSTM + Single Attention

Second model trained, on UCI-HAR's official split. Same structure as
`02_baseline_bigru.ipynb`: a diagnostic single-seed run, then the official
multi-seed benchmark that appends to `raw_results.csv` — plus this notebook
adds the thing Baseline 2 exists to demonstrate: an explicit comparison
against Baseline 1, and a look at what the attention mechanism actually
attends to.

This reproduces the "attention-based BiLSTM" approach from literature review
rows 1/2 — a *single* attention axis (temporal only), as distinct from the
proposed model's *dual* (channel + temporal) attention. Keeping that
distinction clean here is what makes the eventual ablation story legible:
no-attention -> one-attention-axis -> two-attention-axes.

Prerequisite: `python -m src.preprocessing` has populated
`data/processed/ucihar/`, and ideally `02_baseline_bigru.ipynb` has already
run (so `raw_results.csv` has Baseline 1 numbers to compare against — if not,
the comparison cell below will just show Baseline 2 alone).


## Setup

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from pathlib import Path
from sklearn.metrics import confusion_matrix

from src.data_loader import get_official_split, build_dataloaders
from src.train import build_model, train_one_run
from src.utils import (
    set_seed, get_device, encode_labels, profile_model,
    compute_metrics, append_result_row, save_checkpoint,
)

MODEL_NAME = "bilstm_attention_baseline"
DATASET = "ucihar"
device = get_device()
print("device:", device)


## Part 1 — Diagnostic single-seed run

In [ ]:
X_train, y_train, X_test, y_test = get_official_split(DATASET)
y_train_enc, y_test_enc, mapping = encode_labels(y_train, y_test)
num_classes = len(mapping)
input_shape = X_train.shape[1:]
print(f"train {X_train.shape}, test {X_test.shape}, num_classes={num_classes}")

set_seed(0)
model = build_model(MODEL_NAME, input_size=input_shape[-1], num_classes=num_classes).to(device)
train_loader, test_loader = build_dataloaders(X_train, y_train_enc, X_test, y_test_enc, batch_size=64)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

EPOCHS = 50
train_losses, val_losses = [], []
for epoch in range(EPOCHS):
    model.train()
    running = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        running += loss.item()
    train_losses.append(running / len(train_loader))

    model.eval()
    running = 0.0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            running += criterion(model(xb), yb).item()
    val_losses.append(running / len(test_loader))

    if (epoch + 1) % 10 == 0:
        print(f"epoch {epoch+1:>3}  train_loss={train_losses[-1]:.4f}  val_loss={val_losses[-1]:.4f}")


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(train_losses, label="train")
plt.plot(val_losses, label="val (test set)")
plt.xlabel("epoch")
plt.ylabel("cross-entropy loss")
plt.title(f"{MODEL_NAME} on {DATASET} — training curve")
plt.legend()
plt.tight_layout()
plt.show()


## Confusion matrix

In [ ]:
model.eval()
all_preds, all_true = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        preds = model(xb.to(device)).argmax(dim=1).cpu().numpy()
        all_preds.append(preds)
        all_true.append(yb.numpy())
y_pred = np.concatenate(all_preds)
y_true = np.concatenate(all_true)

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xlabel("predicted")
ax.set_ylabel("true")
ax.set_title(f"{MODEL_NAME} on {DATASET} — confusion matrix")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                 color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.colorbar(im)
plt.tight_layout()
plt.show()

print(compute_metrics(y_true, y_pred))


Compare this confusion matrix's off-diagonal pattern against
`02_baseline_bigru.ipynb`'s — the walking/walking-upstairs confusion flagged
in the literature review (row 2) is specifically the kind of error attention
is supposed to help with. Worth eyeballing whether that pair's confusion
count actually went down here relative to Baseline 1, not just whether
overall accuracy improved.

## Attention weight inspection

What timesteps within a 2.56s window does the trained model attend to?
Compared later against the Dual-Attention BiGRU's temporal attention in
notebook 04 — if the two models converge on similar temporal patterns, that's
mild evidence the *temporal* half of dual attention isn't adding much beyond
what single attention already captures, which would be a legitimate,
reportable ablation finding either way.

In [ ]:
model.eval()
sample_x, sample_y = next(iter(test_loader))
with torch.no_grad():
    logits, weights = model(sample_x.to(device), return_attention=True)

fig, axes = plt.subplots(1, 3, figsize=(15, 3), sharey=True)
for ax, i in zip(axes, range(3)):
    ax.plot(weights[i].cpu().numpy())
    ax.set_title(f"sample {i} (true label={sample_y[i].item()})")
    ax.set_xlabel("timestep")
axes[0].set_ylabel("attention weight")
plt.tight_layout()
plt.show()


## Part 2 — Official multi-seed benchmark run

In [ ]:
SEEDS = [0, 1, 2]
RESULTS_CSV = Path("../results/tables/raw_results.csv")
CHECKPOINT_DIR = Path("../checkpoints") / MODEL_NAME

probe = build_model(MODEL_NAME, input_size=input_shape[-1], num_classes=num_classes)
efficiency = profile_model(probe, input_shape, device)
print("efficiency:", efficiency)

run_metrics = []
for seed in SEEDS:
    set_seed(seed)
    model = build_model(MODEL_NAME, input_size=input_shape[-1], num_classes=num_classes)
    train_loader, test_loader = build_dataloaders(X_train, y_train_enc, X_test, y_test_enc, batch_size=64)
    metrics = train_one_run(model, train_loader, test_loader, device, epochs=50, lr=1e-3, patience=8)
    print(f"seed {seed}: {metrics}")
    run_metrics.append(metrics)

    save_checkpoint(model, CHECKPOINT_DIR / f"{DATASET}_official_seed{seed}.pt")
    append_result_row({
        "model": MODEL_NAME, "dataset": DATASET, "split_mode": "official",
        "fold_id": None, "seed": seed, **metrics, **efficiency,
    }, csv_path=RESULTS_CSV)

acc = [m["accuracy"] for m in run_metrics]
f1 = [m["macro_f1"] for m in run_metrics]
print(f"\n{MODEL_NAME} official split -- accuracy: {np.mean(acc):.4f} +/- {np.std(acc):.4f}, "
      f"macro-F1: {np.mean(f1):.4f} +/- {np.std(f1):.4f}")


## Compare against Baseline 1

Reads back everything logged to `raw_results.csv` so far for these two
models and plots them side by side. If `02_baseline_bigru.ipynb` hasn't been
run yet, only Baseline 2 will show up here — that's expected, not an error.


In [ ]:
df = pd.read_csv(RESULTS_CSV)
df = df[(df["dataset"] == DATASET) & (df["split_mode"] == "official") &
        (df["model"].isin(["bigru_baseline", "bilstm_attention_baseline"]))]

summary = df.groupby("model")[["accuracy", "macro_f1"]].agg(["mean", "std"])
display(summary)

if "bigru_baseline" in df["model"].unique():
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, metric in zip(axes, ["accuracy", "macro_f1"]):
        means = df.groupby("model")[metric].mean()
        stds = df.groupby("model")[metric].std()
        ax.bar(means.index, means.values, yerr=stds.values, capsize=5)
        ax.set_title(metric)
        ax.tick_params(axis="x", rotation=20)
    plt.tight_layout()
    plt.show()
else:
    print("Baseline 1 not in raw_results.csv yet -- run 02_baseline_bigru.ipynb first for the full comparison.")


## Notes / next steps

- [ ] Did single temporal attention improve accuracy/macro-F1 over the plain
      BiGRU baseline, and by how much? This is the number to cite when later
      arguing dual attention is worth its extra params over single attention.
- [ ] Checkpoints saved to `checkpoints/bilstm_attention_baseline/ucihar_official_seed{0,1,2}.pt`.
- [ ] `params` for this model should come out higher than `bigru_baseline`'s
      (extra attention MLP + BiLSTM vs BiGRU) — worth keeping in mind for the
      eventual accuracy-vs-efficiency scatter plot in notebook 04.
- [ ] Next: `cnn_bigru_reference.py` (Week 6) is already implemented; its own
      notebook isn't in the plan's structure, so it gets its first real
      numbers directly in `04_dual_attention_bigru.ipynb`'s all-4-models pass.
